# Code mẫu — Language model

Language model gán xác suất cho token tiếp theo. Decoding là bước chọn một đường đi qua chuỗi các phân phối ấy để sinh ra văn bản. Các ví dụ dưới đây dùng một kho câu tự soạn và các logit đặt bằng tay, để kiểm từng phần: đếm, tính loss, chọn token, và điều kiện dừng.

## Ví dụ 1: Đếm bigram, và thấy tác dụng của smoothing

Năm câu train dùng “em” và “bạn” làm chủ ngữ. Sau “em”, chữ “học” xuất hiện hai lần, chữ “đọc” một lần. Từ điển các token có thể đứng sau gồm mọi chữ trong train, cộng thêm `EOS` đánh dấu hết câu và `UNK` cho chữ lạ. `BOS` đánh dấu đầu câu, chỉ đứng trước nên không có trong từ điển này.

In [ ]:
from collections import Counter, defaultdict
import numpy as np

train = ["em học toán", "em học tin", "em đọc sách", "bạn học toán", "bạn đọc sách"]
BOS, EOS, UNK = "<bos>", "<eos>", "<unk>"
vocabulary = sorted({word for sentence in train for word in sentence.split()} | {EOS, UNK})
counts = defaultdict(Counter)
for sentence in train:
    tokens = [BOS] + sentence.split() + [EOS]
    for previous, following in zip(tokens, tokens[1:]):
        counts[previous][following] += 1

def distribution(previous, alpha=1.):
    if alpha <= 0:
        raise ValueError("Demo yêu cầu alpha dương")
    row = counts.get(previous, Counter())
    return np.array([(row[word] + alpha) / (sum(row.values()) + alpha * len(vocabulary))
                     for word in vocabulary])

print("số token đích:", len(vocabulary))
print("sau em:", dict(counts["em"]))
print("không smoothing, p(học|em):", round(2 / 3, 3))
for alpha in [0.1, 1.]:
    probability = distribution("em", alpha)
    print(f"alpha={alpha}: p(học|em)={probability[vocabulary.index('học')]:.3f} tổng={probability.sum():.3f}")
    assert np.isclose(probability.sum(), 1)
assert np.isclose(distribution("em")[vocabulary.index("học")], 3 / 12)
assert np.all(distribution(UNK) > 0)

số token đích: 9
sau em: {'học': 2, 'đọc': 1}
không smoothing, p(học|em): 0.667
alpha=0.1: p(học|em)=0.538 tổng=1.000
alpha=1.0: p(học|em)=0.250 tổng=1.000


Không smoothing, xác suất “học” sau “em” là 2/3. Mọi chữ chưa từng đứng sau “em”, như “toán”, có xác suất 0. Smoothing cộng thêm alpha vào mọi số đếm để không còn xác suất 0. Với alpha bằng 1, tử số là $2+1=3$, mẫu số là $3+9=12$, nên xác suất “học” chỉ còn 0.25.

Với kho câu nhỏ như thế này, alpha lớn làm phân phối gần như phẳng, xóa mất những gì đã học. Alpha nhỏ hơn giữ được nhiều thông tin hơn. Tăng alpha không tự làm model tốt lên, phải chọn bằng dữ liệu kiểm tra.

Tiếp theo, ta tính NLL và perplexity cho ba câu thử. NLL là trung bình của âm log xác suất mỗi token. Perplexity, gọi tắt là PPL, bằng $e^{\text{NLL}}$. Token `EOS` cũng được tính, vì model phải đoán cả lúc dừng. Chữ “robot” không có trong từ điển, nên được đổi thành `UNK`.

In [ ]:
def sentence_nll(sentence):
    words = [word if word in vocabulary else UNK for word in sentence.split()]
    sequence = [BOS] + words + [EOS]
    probabilities = [distribution(left)[vocabulary.index(right)]
                     for left, right in zip(sequence, sequence[1:])]
    return -np.log(probabilities).mean(), len(probabilities)

for sentence in ["em học toán", "em sách học", "robot học toán"]:
    nll, token_count = sentence_nll(sentence)
    print(f"{sentence}: N={token_count} NLL={nll:.3f} PPL={np.exp(nll):.3f}")
assert sentence_nll("em học toán")[0] < sentence_nll("em sách học")[0]

em học toán: N=4 NLL=1.331 PPL=3.785
em sách học: N=4 NLL=2.155 PPL=8.629
robot học toán: N=4 NLL=1.880 PPL=6.557


Mỗi câu có ba chữ, cộng `EOS`, nên có 4 token được chấm. Câu đúng trật tự có PPL 3.785, thấp hơn nhiều so với 8.629 của câu đảo trật tự. Model “ít bất ngờ” hơn với câu quen. Câu đầu nằm sẵn trong train, nên đây chỉ là phép kiểm cách tính. Nó chưa phải điểm đánh giá độc lập, và PPL cũng không phải công cụ để đoán ai đã viết một đoạn văn.

## Ví dụ 2: Dịch nhãn lệch một vị trí, và bỏ PAD khỏi mẫu số

Hai câu có độ dài khác nhau: `BOS em học EOS` và `BOS em EOS`. Ta đệm cho cùng độ dài. Đầu vào là chuỗi bỏ token cuối, nhãn là chuỗi bỏ token đầu, nên mỗi vị trí học đoán token ngay sau nó. Ta in đầu vào và nhãn, rồi tính loss trên đúng những token thật.

In [ ]:
import math
import torch
from torch import nn

torch.set_num_threads(1)
names = ["PAD", "BOS", "EOS", "em", "học"]
PAD = 0
sequences = torch.tensor([[1, 3, 4, 2, 0], [1, 3, 2, 0, 0]])
inputs = sequences[:, :-1]
labels = sequences[:, 1:].clone()
labels[labels == PAD] = -100
for x, y in zip(inputs.tolist(), labels.tolist()):
    print("input:", [names[i] for i in x])
    print("label:", [names[i] if i != -100 else "bỏ" for i in y])
allowed = torch.ones(4, 4, dtype=torch.bool).tril()
print("causal: 1 là được đọc:", allowed.int().tolist())

# Logit lấy từ một phân phối đặt sẵn: ở mọi vị trí thật, lớp đúng có xác suất 1/4.
probabilities = torch.full((2, 4, 5), 0.75 / 4, dtype=torch.float64)
for row, position in labels.ne(-100).nonzero().tolist():
    probabilities[row, position, labels[row, position]] = 0.25
logits = probabilities.log()
loss = nn.CrossEntropyLoss(ignore_index=-100)(logits.reshape(-1, 5), labels.reshape(-1))
print("token được chấm:", int(labels.ne(-100).sum()))
print(f"NLL={loss.item():.3f} PPL={math.exp(loss.item()):.3f}")
assert int(labels.ne(-100).sum()) == 5
assert math.isclose(loss.item(), math.log(4), abs_tol=1e-10)
assert not allowed[0, 1] and allowed[2, 2]

input: ['BOS', 'em', 'học', 'EOS']
label: ['em', 'học', 'EOS', 'bỏ']
input: ['BOS', 'em', 'EOS', 'PAD']
label: ['em', 'EOS', 'bỏ', 'bỏ']
causal: 1 là được đọc: [[1, 0, 0, 0], [1, 1, 0, 0], [1, 1, 1, 0], [1, 1, 1, 1]]
token được chấm: 5
NLL=1.386 PPL=4.000


Đọc câu đầu theo từng cột. Ở vị trí có `BOS`, nhãn là `em`. Ở vị trí có `học`, nhãn là `EOS`. Ở vị trí có `EOS`, nhãn là PAD, nên bị bỏ: câu đã kết thúc, không cần đoán gì thêm. Cả hai câu có tổng cộng 5 token được chấm, chứ không phải 8. Mỗi token đúng có xác suất 1/4, nên NLL bằng $\ln 4$ và PPL bằng 4.

Ma trận causal cho mỗi vị trí đọc chính nó và các vị trí trước, không đọc vị trí sau. Đọc chính nó là được, vì nhãn ở đó là token tiếp theo, chứ không phải token hiện tại. Logit trong ví dụ được đặt sẵn để kiểm loss, chưa chạy qua Transformer nào.

## Ví dụ 3: Temperature đổi xác suất, top-k và top-p đổi tập được chọn

Ba token A, B, C có logit `(2, 1, 0)`. Ta so ba mức temperature, rồi áp top-k với k = 2 và top-p với p = 0.8 lên cùng phân phối gốc. Top-k giữ k token có xác suất cao nhất. Top-p giữ ít token nhất sao cho tổng xác suất đạt ngưỡng p.

In [ ]:
import numpy as np

def softmax_temperature(logits, temperature):
    if temperature <= 0:
        raise ValueError("Temperature phải dương khi chia logits")
    scaled = np.asarray(logits, dtype=float) / temperature
    weights = np.exp(scaled - scaled.max())
    return weights / weights.sum()

def retain(probability, indices):
    filtered = np.zeros_like(probability)
    filtered[indices] = probability[indices]
    return filtered / filtered.sum()

logits = np.array([2., 1., 0.])
for temperature in [0.5, 1., 2.]:
    p = softmax_temperature(logits, temperature)
    print(f"T={temperature}: {p.round(3).tolist()} greedy={int(p.argmax())}")
base = softmax_temperature(logits, 1.)
order = np.argsort(-base, kind="stable")
top_k = retain(base, order[:2])
count = int(np.searchsorted(np.cumsum(base[order]), 0.8, side="left")) + 1
top_p = retain(base, order[:count])  # Giữ luôn token làm tổng vừa chạm hoặc vượt ngưỡng.
print("top-k 2:", top_k.round(3).tolist())
print("top-p 0.8:", top_p.round(3).tolist(), "số giữ:", count)
assert np.isclose(top_p.sum(), 1)
assert count == 2 and np.array_equal(top_k, top_p)
assert softmax_temperature(logits, 0.5).argmax() == softmax_temperature(logits, 2).argmax()

T=0.5: [0.867, 0.117, 0.016] greedy=0
T=1.0: [0.665, 0.245, 0.09] greedy=0
T=2.0: [0.506, 0.307, 0.186] greedy=0
top-k 2: [0.731, 0.269, 0.0]
top-p 0.8: [0.731, 0.269, 0.0] số giữ: 2


Tăng temperature từ 0.5 lên 2, xác suất của A giảm từ 0.867 xuống 0.506, phân phối phẳng dần. Nhưng greedy, tức luôn chọn token cao nhất, vẫn chọn A, vì thứ tự các logit không đổi. Temperature chỉ có tác dụng khi ta lấy mẫu ngẫu nhiên.

Ở bộ số này, top-k và top-p tình cờ giữ cùng hai token. Với phân phối `(0.45, 0.30, 0.15, 0.10)` thì khác: hai token đầu mới được 0.75, chưa tới 0.8, nên top-p phải giữ ba token. Top-p tự co giãn theo độ chắc chắn của phân phối, còn top-k thì luôn giữ đúng k. Dù lọc cách nào, token được chọn vẫn có thể sai về nội dung.

## Ví dụ 4: Beam search giữ lại đường đi mà greedy bỏ sớm

Ta dựng một cây xác suất nhỏ, đủ nhỏ để liệt kê hết mọi câu có thể sinh ra. Ở bước đầu, A thắng với 0.6. Nhưng sau B thì xác suất kết thúc bằng `EOS` cao hơn hẳn. Ta chạy greedy, chạy beam search giữ hai ứng viên, và duyệt hết mọi đường để có đáp án chuẩn. Tối đa ba bước sinh.

In [ ]:
import math

transitions = {
    "BOS": [("A", 0.6), ("B", 0.4)],
    "A": [("EOS", 0.5), ("C", 0.5)],
    "B": [("EOS", 0.95), ("C", 0.05)],
    "C": [("EOS", 1.0)],
}
assert all(math.isclose(sum(p for _, p in row), 1) for row in transitions.values())

def beam_search(width, max_steps=3):
    beams = [((), 0.)]  # Cộng log thay vì nhân xác suất, để tránh tràn số dưới.
    for _ in range(max_steps):
        candidates = []
        for tokens, score in beams:
            if tokens and tokens[-1] == "EOS":
                candidates.append((tokens, score))
                continue
            previous = tokens[-1] if tokens else "BOS"
            for token, probability in transitions[previous]:
                candidates.append((tokens + (token,), score + math.log(probability)))
        beams = sorted(candidates, key=lambda item: -item[1])[:width]
        if all(tokens[-1] == "EOS" for tokens, _ in beams):
            break
    return beams[0]

def all_finished(tokens=(), probability=1.):
    if tokens and tokens[-1] == "EOS":
        return [(tokens, probability)]
    if len(tokens) == 3:
        return []
    previous = tokens[-1] if tokens else "BOS"
    return [answer for token, p in transitions[previous]
            for answer in all_finished(tokens + (token,), probability * p)]

for width in [1, 2]:
    tokens, log_probability = beam_search(width)
    print(f"beam={width}: {' '.join(tokens)} p={math.exp(log_probability):.3f}")
oracle = max(all_finished(), key=lambda item: item[1])
print("duyệt hết:", " ".join(oracle[0]), "p=", round(oracle[1], 3))
assert beam_search(1)[0] == ("A", "EOS")
assert beam_search(2)[0] == oracle[0] == ("B", "EOS")

beam=1: A EOS p=0.300
beam=2: B EOS p=0.380
duyệt hết: B EOS p= 0.38


Beam rộng 1 chính là greedy. Nó chọn A ở bước đầu, rồi chỉ đạt $0.6\cdot0.5=0.30$. Sau A, hai lựa chọn `EOS` và `C` hòa điểm, và `EOS` được giữ vì đứng trước trong bảng. Beam rộng 2 giữ lại cả B, nên tìm ra câu `B EOS` với xác suất $0.4\cdot0.95=0.38$, khớp với kết quả duyệt hết.

Khớp đáp án trên một cây nhỏ không có nghĩa beam search luôn tìm ra câu tốt nhất. Với cây lớn, beam vẫn có thể bỏ sót. Ngoài ra, nếu thêm phạt theo độ dài câu, tiêu chí xếp hạng sẽ khác, và câu thắng cũng có thể khác.